In [2]:
from fastapi import APIRouter, Query, HTTPException
from typing import Optional, List, Dict, Any, Literal
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import json
import re
import sys

sys.path.insert(0, r"D:/A0_Project")  # 讓它優先被找到
from PI_SYSTEM.models.sql_db_connect import MySQLConnet
dbhandler = MySQLConnet('piaoi_density')
cim_dbhandler = MySQLConnet('cim_piaoi')
old_dbhandler = MySQLConnet('l6a01_project')

from PI_SYSTEM.models.density.cim_density_job import Config as DensityJobConfig
from PI_SYSTEM.models.density.API_Config import API_Config

In [6]:
tbn = 'cim_defect_202603_aoi200_capic500'
tbn = 'cim_pi_glass_202603'
df = cim_dbhandler.get_runs_delta_days(tbn, 3, 'test_time')
df.head()

,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
0,AJ5R130WBK,2026-03-31 14:00:38,M195RTN01_RG,PI AOI,CF,803,CA0189,CAPIT203,80,0,0,24,56,CAPIC200,2026-03-31 11:08:46,2026-03-31 10:00:00,API
1,AJ5R130WA5,2026-03-31 13:57:15,M195RTN01_RG,PI AOI,CF,803,CA0189,CAPIT203,80,0,3,27,50,CAPIC200,2026-03-31 11:08:46,2026-03-31 10:00:00,API
2,AJ5R130W9G,2026-03-31 13:54:06,M195RTN01_RG,PI AOI,CF,803,CA0189,CAPIT203,56,0,0,20,36,CAPIC200,2026-03-31 11:08:46,2026-03-31 10:00:00,API
3,AJ5R130W9S,2026-03-31 13:50:50,M195RTN01_RG,PI AOI,CF,803,CA0189,CAPIT203,73,0,0,24,49,CAPIC200,2026-03-31 11:08:46,2026-03-31 10:00:00,API
4,AJ5R130W9K,2026-03-31 13:40:53,M195RTN01_RG,PI AOI,CF,803,CA0189,CAPIT203,115,0,3,46,66,CAPIC200,2026-03-31 11:08:46,2026-03-31 10:00:00,API


In [7]:
df.iloc[0,:].to_dict()

{'sheet_id_chip_id': 'AJ5R130WBK',
 'test_time': Timestamp('2026-03-31 14:00:38'),
 'model_no': 'M195RTN01_RG',
 'op_id': 'PI AOI',
 'abbr_cat': 'CF',
 'recipe_id': '803',
 'cassette_id': 'CA0189',
 'aoi': 'CAPIT203',
 'total_defect_qty': 80,
 'defect_size_o_qty': 0,
 'defect_size_l_qty': 0,
 'defect_size_m_qty': 24,
 'defect_size_s_qty': 56,
 'line_id': 'CAPIC200',
 'pi_time': Timestamp('2026-03-31 11:08:46'),
 'pi_hour': Timestamp('2026-03-31 10:00:00'),
 'pi_type': 'API'}

In [2]:
tbn = "aoi_inspection_default_spectable"
tb = old_dbhandler.get_table(tbn)
tb

,model,glass_type,line_id,OOC,OOS,Editor,modify_time,drop
0,A101EAN01,TFT,CAPIC107,0.5,0.8,預設,2026-01-02 13:52:53,F
1,B070AAN01,TFT,CAPIC107,0.5,0.8,預設,2025-12-24 09:00:00,F
2,B070AAN1Y,TFT,CAPIC107,0.5,0.8,預設,2025-12-24 09:00:00,F
3,B080EAN05,TFT,CAPIC107,0.5,0.8,預設,2025-12-24 09:00:00,F
4,B101EAN02,TFT,CAPIC107,0.5,0.8,預設,2025-12-24 09:00:00,F
...,...,...,...,...,...,...,...,...
724,M340QAR1A,TFT,CAPIC607,0.5,0.8,預設,2026-01-05 08:54:55,F
725,M270QAN06,TFT,CAPIC207,3.0,4.0,預設,2026-01-15 11:46:34,F
726,M270QAN06,TFT,CAPIC507,2.0,3.0,預設,2026-02-21 11:34:13,F
727,M270DANA0,TFT,CAPIC407,2.0,3.0,預設,2026-03-04 08:12:10,F


In [ ]:
db = MySQLConnet('piaoi_inspection_density')
tbn = "inspection_api_summary_202604"
#tbn = "default_spec_table"
#rows = db.get_runs_delta_days(tbn, 2, 'pi_hour')
#rows
tb[tb['model'].str.contains("test")]
tb = db.get_table(tbn)
print(len(tb))


1079


In [26]:
#db = MySQLConnet('piaoi_inspection_density')
db = MySQLConnet('piaoi_inspection_density')
rdf = db.get_table('inspection_raw_table_202604')
sdf = db.get_table('inspection_summary_table_202604')

In [25]:
old_df = old_dbhandler.get_table('inspection_raw_table_202604')
db.save_table('inspection_raw_table_202604', old_df)

2026-04-08 11:16:58,768 - INFO - [save_table] piaoi_inspection_density.inspection_raw_table_202604 已寫入 201734 列（去除 0 重複）


201734

In [27]:
non_id = 0
non_sheet = 0
same1 = 0
same2 = 0
non_df = pd.DataFrame()
sheet_id_non_df = pd.DataFrame()
for (runid, sheet, ), s_raw in sdf.groupby(['RUN_ID', 'SHEET_ID']):
    s_len = float(s_raw['TOTAL_DEFECT_COUNT'].unique()[0])
    if s_len ==0: continue
    raw = rdf[(rdf['RUN_ID'] == runid) & (rdf['SHEET_ID'] == sheet)]
    
    if not raw.empty:
        
        if len(raw) != s_len and len(raw)!=1000:
            print("defect count fail")
            print(runid, sheet)
            print( s_raw)
            print(len(raw))
            print()
            print(raw)
            non_df = pd.concat([non_df, s_raw])
            same1 +=1
        else:
            same2 +=1
    else:
        print('runid fail: ', runid, sheet)
        print( s_raw)
        raw = rdf[(rdf['SHEET_ID'] == sheet)]
        if not raw.empty:
            print("-----runid fail 'SHEET_ID' ok ----")
            
            print(len(raw))
            print(raw)
            non_id +=1
        else:
            print("-----only 'SHEET_ID' also fail----")
            sheet_id_non_df = pd.concat([sheet_id_non_df, s_raw])
            non_sheet +=1
    #print('=====')

runid fail:  -2116673833.0 AA1EZ4020H
      CHIP_COUNT CHIP_JUDGE CHIP_OK_COUNT DEFECT  FAB   MODEL_NO  \
37604       71.0         OK          71.0   True  L6A  G101EAN2G   

        RECIPE_NAME         RUN_ID        SCAN_ENDTIME  \
37604  G101EAN2G-CF  -2116673833.0 2026-04-08 10:00:41   

                SCAN_STARTTIME    SHEET_ID STAGE   TOOL_ID TOTAL_DEFECT_COUNT  \
37604  2026-04-08 10:00:18.000  AA1EZ4020H  CELL  CAPIC107                1.0   

      TYPE  
37604   CF  
-----only 'SHEET_ID' also fail----
runid fail:  -2116674245.0 AA1EZ4024V
      CHIP_COUNT CHIP_JUDGE CHIP_OK_COUNT DEFECT  FAB   MODEL_NO  \
37183       71.0         OK          71.0   True  L6A  G101EAN2G   

        RECIPE_NAME         RUN_ID        SCAN_ENDTIME  \
37183  G101EAN2G-CF  -2116674245.0 2026-04-08 08:42:07   

                SCAN_STARTTIME    SHEET_ID STAGE   TOOL_ID TOTAL_DEFECT_COUNT  \
37183  2026-04-08 08:41:45.000  AA1EZ4024V  CELL  CAPIC107                1.0   

      TYPE  
37183   CF  
---

In [ ]:
[{"run_id": "-2116686255.0", "glass_id": "20260405163152", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686250.0", "glass_id": "N46A2NX51", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686249.0", "glass_id": "N46A2NX52", "summary_defect_count": 2, "raw_defect_count": 2, "S": 0, "M": 1, "L": 1, "O": 0, "raw_matched": true}, {"run_id": "-2116686210.0", "glass_id": "N46A2NX53", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686209.0", "glass_id": "N46A2NX54", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686243.0", "glass_id": "N46A2NX5X", "summary_defect_count": 2, "raw_defect_count": 2, "S": 1, "M": 1, "L": 0, "O": 0, "raw_matched": true}, {"run_id": "-2116686242.0", "glass_id": "N46A2NX5Y", "summary_defect_count": 5, "raw_defect_count": 5, "S": 1, "M": 2, "L": 0, "O": 2, "raw_matched": true},
  {"run_id": "-2116686055.0", "glass_id": "N46A3JG8A", "summary_defect_count": 1, "raw_defect_count": 1, "S": 0, "M": 1, "L": 0, "O": 0, "raw_matched": true}, {"run_id": "-2116686054.0", "glass_id": "N46A3JG8B", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686053.0", "glass_id": "N46A3JG8C", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686052.0", "glass_id": "N46A3JG8D", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}, {"run_id": "-2116686051.0", "glass_id": "N46A3JG8E", "summary_defect_count": 0, "raw_defect_count": 0, "S": 0, "M": 0, "L": 0, "O": 0, "raw_matched": false}]

In [28]:
print(non_id ,non_sheet ,same1,same2)

0 44 23 16232


In [2]:
d_tbns = dbhandler.list_tables()
print(d_tbns)

c_tbns = cim_dbhandler.list_tables()
c_tbns[:3]

o_tbns = old_dbhandler.list_tables()
[v for v in o_tbns if 'spec_' in v]

2026-02-12 14:51:44,203 - INFO - [list_tables] 成功取得資料表名稱，共 4 張表。
2026-02-12 14:51:44,208 - INFO - [list_tables] 成功取得資料表名稱，共 69 張表。
2026-02-12 14:51:44,236 - INFO - [list_tables] 成功取得資料表名稱，共 205 張表。


['default_spec_table', 'density_summary_202601', 'density_summary_202602', 'fix_spec_table_202602']


['aoi_density_fixed_spec_table',
 'aoi_density_spec_for_aoimonitor',
 'aoi_spec_for_aoimonitor',
 'spec_pro_update__table',
 'spec_pro_update_history']

In [8]:
60*24

1440

In [4]:
#spec_df = old_dbhandler.get_table('aoi_density_spec_for_aoimonitor')
#spec_df[(spec_df['line_id'] == 'CAPIC300') & (spec_df['MODEL_ID'] == 'Z335XAN01') ]
spec_df = dbhandler.get_table('fix_spec_table_202602')
spec_df.iloc[-1,:].to_dict()

{'line_id': 'CAPIC200',
 'aoi': 'aoi100',
 'model': 'G215HVN1C',
 'recipe_id': '608',
 'glass_type': 'CF',
 'adc_def_code': 'SPS',
 'defect_size': 'OLM',
 'total_glass_cnt': 28,
 'defect_cnt': 0.0,
 'density': 0.0,
 'overD': 0.0,
 'removed_glasses': 0,
 'removed_defects': 0.0,
 'final_glass_cnt': 28,
 'final_defect_cnt': 0.0,
 'final_density': 0.0,
 'std': 0.0,
 'OOC': 0.0,
 'OOS': 0.0,
 'GEN_DT': Timestamp('2026-02-10 08:00:03'),
 'PERIOD_START': Timestamp('2025-11-12 07:30:00'),
 'PERIOD_END': Timestamp('2026-02-10 07:30:00'),
 'DAYS': 90}

In [2]:
c_tb = cim_dbhandler.get_table('cim_pi_glass_202601')#'cim_defect_202512_aoi100_capic100'
print(c_tb.iloc[0,:].to_dict())
d_tb = dbhandler.get_table('density_summary_202602')
print(len(d_tb))
d_tb.iloc[0,:].to_dict()

{'sheet_id_chip_id': 'YM5ACMK5A', 'test_time': Timestamp('2025-12-24 00:15:34'), 'model_no': 'B156HAN2U', 'op_id': 'KOUID', 'abbr_cat': 'TFT', 'recipe_id': '4309', 'cassette_id': 'AA0023', 'aoi': 'CAAOI202', 'total_defect_qty': 81, 'defect_size_o_qty': 0, 'defect_size_l_qty': 1, 'defect_size_m_qty': 28, 'defect_size_s_qty': 52, 'line_id': 'CAPIC100', 'pi_time': Timestamp('2026-01-05 22:46:41'), 'pi_hour': Timestamp('2026-01-05 22:00:00'), 'pi_type': 'BPI'}
2757


{'line_id': 'CAPIC100',
 'aoi': 'aoi200',
 'model': 'B156HAN2U',
 'glass_type': 'TFT',
 'recipe_id': '0309',
 'pi_hour': Timestamp('2026-02-01 03:00:00'),
 'adc_def_code': 'NPI_OIL',
 'defect_cnt': 4,
 'glass_cnt': 7,
 'density': 0.571,
 'def_glass_cnt': 2,
 'small_defect_count': 1,
 'middle_defect_count': 3,
 'large_defect_count': 0,
 'over_defect_count': 0,
 'glass': 'YL6A1Q63A,YL6A1Q63B,YL6A1Q63C,YL6A1Q63D,YL6A1Q63E,YL6A1Q63F,YL6A1Q63G',
 'glass_size_detail': '{"YL6A1Q63A": {"S": 0, "M": 1, "L": 0, "O": 0, "T": 1}, "YL6A1Q63B": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "YL6A1Q63C": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "YL6A1Q63D": {"S": 1, "M": 2, "L": 0, "O": 0, "T": 3}, "YL6A1Q63E": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "YL6A1Q63F": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}, "YL6A1Q63G": {"S": 0, "M": 0, "L": 0, "O": 0, "T": 0}}',
 'shift': '{}',
 'comment': '',
 'action': '',
 'Editor': '',
 'modify_time': '2026-02-06 13:54:13'}

In [ ]:
d_tb.columns
cols = ['line_id', 'aoi', 'model', 'glass_type', 'recipe_id', 'pi_hour','adc_def_code', 'density',
  'comment', 'action', 'Editor', 'modify_time',
       ]
d_tb[cols]

,line_id,aoi,model,glass_type,recipe_id,pi_hour,adc_def_code,density,comment,action,Editor,modify_time
0,CAPIC100,aoi200,B156HAN2U,TFT,0309,2026-02-01 03:00:00,NPI_OIL,0.571,,,,2026-02-06 13:54:13
1,CAPIC100,aoi200,B156HAN2U,TFT,0309,2026-02-01 03:00:00,NPI_TFT,0.429,,,,2026-02-06 13:54:13
2,CAPIC100,aoi200,B156HAN2U,TFT,0309,2026-02-01 03:00:00,Not_Found,1.429,,,,2026-02-06 13:54:13
3,CAPIC100,aoi200,B156HAN2U,TFT,0309,2026-02-01 03:00:00,PIS With Particle,1.286,,,,2026-02-06 13:54:13
4,CAPIC100,aoi200,B156HAN2U,TFT,0309,2026-02-01 03:00:00,PI_Spot_NP,0.143,,,,2026-02-06 13:54:13
...,...,...,...,...,...,...,...,...,...,...,...,...
2752,CAPIC700,aoi200,M270QAN06,TFT,4234,2026-02-07 05:00:00,Not_Found,0.357,,,,2026-02-07 16:05:48
2753,CAPIC700,aoi200,M270QAN06,TFT,4234,2026-02-07 05:00:00,OP_Defect,26.429,,,,2026-02-07 16:05:48
2754,CAPIC700,aoi200,M270QAN06,TFT,4234,2026-02-07 05:00:00,PI_Gel,0.143,,,,2026-02-07 16:05:48
2755,CAPIC700,aoi200,M270QAN06,TFT,4234,2026-02-07 05:00:00,Polymer,0.071,,,,2026-02-07 16:05:48


In [13]:
group_cols = ["line_id", "aoi",'model_no', "abbr_cat", "recipe_id", "pi_hour"]
sort_cols = ["pi_time", "test_time"]
order_cols = group_cols + ["sheet_id_chip_id"] + sort_cols
tmp = c_tb.copy()
tmp = tmp.sort_values(order_cols, ascending=True)
# per (group + glass) keep latest
tmp = tmp.drop_duplicates(subset=group_cols + ["sheet_id_chip_id"], keep="last")
print(f'[keep latest]ori rows: {len(c_tb)}, drop rows: : {len(tmp)}')

[keep latest]ori rows: 17160, drop rows: : 15755


In [11]:
d_tb[d_tb['glass'].str.contains('AP0H11145N')]
c_tb[c_tb['sheet_id_chip_id'].str.contains('AP0H11145N')].to_dict()

{'sheet_id_chip_id': {1155: 'AP0H11145N'},
 'test_time': {1155: Timestamp('2026-02-03 08:20:08')},
 'model_no': {1155: 'G240HW01_VX'},
 'op_id': {1155: 'PI AOI'},
 'abbr_cat': {1155: 'CF'},
 'recipe_id': {1155: '602'},
 'cassette_id': {1155: 'CA0346'},
 'aoi': {1155: 'CAPIT203'},
 'total_defect_qty': {1155: 120},
 'defect_size_o_qty': {1155: 0},
 'defect_size_l_qty': {1155: 2},
 'defect_size_m_qty': {1155: 6},
 'defect_size_s_qty': {1155: 112},
 'line_id': {1155: 'CAPIC200'},
 'pi_time': {1155: Timestamp('2026-02-03 05:59:55')},
 'pi_hour': {1155: Timestamp('2026-02-03 05:00:00')},
 'pi_type': {1155: 'API'}}

In [8]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional, Iterable
class Config:
    # connection
    host: str = "10.97.142.217"
    username: str = "l6a01_user"
    password: str = "l6a01$user"

    # dbs
    src_db: str = "cim_piaoi"        # read only
    out_db: str = "piaoi_density"    # write

    # source tables template
    summary_table_tpl: str = "cim_pi_glass_yyyymm"           # month table
    defect_table_tpl: str = "cim_defect_yyyymm_aoi_line"     # month table

    # output tables template (per month of pi_hour)
    out_table_tpl: str = "density_summary_yyyymm"

    # group columns (after rename)
    group_cols: Tuple[str, ...] = ("line_id", "aoi", "model", "glass_type", "recipe_id", "pi_hour")

    # sort for latest measurement per glass
    sort_cols: Tuple[str, ...] = ("scan_time", "pi_time")  # scan_time == test_time renamed

    # aoi mapping
    aoi_map: Dict[str, str] = field(default_factory=lambda: {
        "CAPIT203": "aoi100",
        "CAAOI202": "aoi200",
        "CAAOI300": "aoi300",
    })

    # summary column mapping (source -> standardized)
    summary_coldict={
        "line_id": "line_id",
        "aoi": "aoi",
        "model_no": "model",
        "abbr_cat": "glass_type",
        "recipe_id": "recipe_id",
        "pi_hour": "pi_hour",
        "pi_type": "pi_type",
        "sheet_id_chip_id": "glass_id",
        "total_defect_qty": "total_defect_count",
        "pi_time": "pi_time",
        "test_time": "scan_time",
    }

    # defect column mapping (source -> standardized)
    defect_coldict: Dict[str, str] = field(default_factory=lambda: {
        "sheet_id_chip_id": "glass_id",
        "chip_id": "chip_name",
        "test_time": "scan_time",
        "defect_size": "defect_size",
        "pox_x1": "x",
        "pox_y1": "y",
        "image_file_path": "pic_path",
        "image_file_name": "pic_name",
        "retype_def_code": "retype_code",
        "adc_def_code": "adc_def_code",
        "pi_time": "pi_time",
        "pi_hour": "pi_hour",
    })

    # query batching
    batch_size: int = 800

    # loop mode
    loop_minutes: int = 10
    # 每次 loop 回看多久（避免同一小時內晚到資料漏掉 -> 覆蓋用）
    lookback_minutes: int = 180  # 建議 >= 120

    # write
    write_out: bool = True
cfg = Config()

In [11]:
60*24

1440

In [8]:
summary = cim_dbhandler.get_table('cim_pi_glass_202602')
summary .head()

,sheet_id_chip_id,test_time,model_no,op_id,abbr_cat,recipe_id,cassette_id,aoi,total_defect_qty,defect_size_o_qty,defect_size_l_qty,defect_size_m_qty,defect_size_s_qty,line_id,pi_time,pi_hour,pi_type
0,AG0UA10P6J,2026-02-01 02:02:35,B160UAN4A_VB,PI AOI,CF,209,CA0081,CAPIT203,6,0,0,1,5,CAPIC700,2026-02-01 01:12:43,2026-02-01 01:00:00,API
1,AG0UA10NUY,2026-02-01 02:05:05,B160UAN4A_VB,PI AOI,CF,209,CA0081,CAPIT203,15,0,2,2,11,CAPIC700,2026-02-01 01:12:43,2026-02-01 01:00:00,API
2,AG0UA10NUU,2026-02-01 02:07:42,B160UAN4A_VB,PI AOI,CF,209,CA0081,CAPIT203,19,2,0,5,12,CAPIC700,2026-02-01 01:12:43,2026-02-01 01:00:00,API
3,AG0UA10NUX,2026-02-01 02:10:19,B160UAN4A_VB,PI AOI,CF,209,CA0081,CAPIT203,15,0,0,2,13,CAPIC700,2026-02-01 01:12:43,2026-02-01 01:00:00,API
4,AG0UA10P6V,2026-02-01 02:12:54,B160UAN4A_VB,PI AOI,CF,209,CA0081,CAPIT203,14,0,0,3,11,CAPIC700,2026-02-01 01:12:43,2026-02-01 01:00:00,API


In [9]:
need_cols = ['pi_time', 'sheet_id_chip_id', 'test_time',  'total_defect_qty', 'pi_hour', 'pi_type']

cast_cols =['line_id', 'aoi','model_no',  'op_id', 'abbr_cat', 'recipe_id', 'cassette_id']
for (main_keys), rows in summary.groupby(cast_cols):
    
    if len(rows['pi_hour'].unique())>1:
        print('old', rows['pi_hour'].unique())
        earliest_tt = rows['pi_hour'].min()
        others = rows[rows['pi_hour'] != earliest_tt]
        summary.loc[others.index, ['pi_hour']] = earliest_tt
        print(summary.loc[others.index, summary.columns]['pi_hour'].unique())



old <DatetimeArray>
['2026-02-01 04:00:00', '2026-02-01 05:00:00']
Length: 2, dtype: datetime64[ns]
<DatetimeArray>
['2026-02-01 04:00:00']
Length: 1, dtype: datetime64[ns]
old <DatetimeArray>
['2026-02-01 04:00:00', '2026-02-01 05:00:00']
Length: 2, dtype: datetime64[ns]
<DatetimeArray>
['2026-02-01 04:00:00']
Length: 1, dtype: datetime64[ns]
old <DatetimeArray>
['2026-02-02 00:00:00', '2026-02-02 02:00:00']
Length: 2, dtype: datetime64[ns]
<DatetimeArray>
['2026-02-02 00:00:00']
Length: 1, dtype: datetime64[ns]
old <DatetimeArray>
['2026-02-02 04:00:00', '2026-02-02 14:00:00']
Length: 2, dtype: datetime64[ns]
<DatetimeArray>
['2026-02-02 04:00:00']
Length: 1, dtype: datetime64[ns]
old <DatetimeArray>
['2026-02-01 05:00:00', '2026-02-03 22:00:00']
Length: 2, dtype: datetime64[ns]
<DatetimeArray>
['2026-02-01 05:00:00']
Length: 1, dtype: datetime64[ns]
old <DatetimeArray>
['2026-02-02 04:00:00', '2026-02-02 14:00:00']
Length: 2, dtype: datetime64[ns]
<DatetimeArray>
['2026-02-02 04:00:

In [10]:
for (main_keys), rows in summary.groupby(cast_cols):
    
    if len(rows['pi_hour'].unique())>1:
        print('old', rows['pi_hour'].unique())

In [9]:
need_cols = ["line_id", "aoi", "model", "glass_type", "recipe_id", "pi_hour","adc_def_code",
              "defect_cnt", "glass_cnt", "density", "def_glass_cnt",
                "glass","glass_size_detail","small_defect_count", "middle_defect_count", "large_defect_count", "over_defect_count"]
df = dbhandler.get_table('density_summary_202601')
df = df[need_cols ]
df.iloc[0,:].to_dict()

{'line_id': 'CAPIC100',
 'aoi': 'aoi100',
 'model': 'B156HTN06',
 'glass_type': 'CF',
 'recipe_id': '812',
 'pi_hour': Timestamp('2026-01-02 05:00:00'),
 'adc_def_code': 'NPI_TFT',
 'defect_cnt': 7,
 'glass_cnt': 1,
 'density': 7.0,
 'def_glass_cnt': 1,
 'glass': 'AF6H6C05RL',
 'glass_size_detail': '{"AF6H6C05RL": {"S": 3, "M": 4, "L": 0, "O": 0, "T": 7}}',
 'small_defect_count': 3,
 'middle_defect_count': 4,
 'large_defect_count': 0,
 'over_defect_count': 0}

,line_id,aoi,model,glass_type,recipe_id,pi_hour,adc_def_code,defect_cnt,glass_cnt,density,...,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,glass,glass_size_detail,comment,action,Editor,modify_time
0,CAPIC100,aoi100,B156HTN06,CF,812,2026-01-02 05:00:00,NPI_TFT,7,1,7.0,...,3,4,0,0,AF6H6C05RL,"{""AF6H6C05RL"": {""S"": 3, ""M"": 4, ""L"": 0, ""O"": 0...",,,,2026-02-04 10:04:16
1,CAPIC100,aoi100,B156HTN06,CF,812,2026-01-02 05:00:00,OP_Defect,2,1,2.0,...,2,0,0,0,AF6H6C05RL,"{""AF6H6C05RL"": {""S"": 2, ""M"": 0, ""L"": 0, ""O"": 0...",,,,2026-02-04 10:04:16
2,CAPIC100,aoi100,B156HTN06,CF,812,2026-01-02 05:00:00,Polymer,2,1,2.0,...,1,1,0,0,AF6H6C05RL,"{""AF6H6C05RL"": {""S"": 1, ""M"": 1, ""L"": 0, ""O"": 0...",,,,2026-02-04 10:04:16
3,CAPIC100,aoi100,B156HTN06,CF,812,2026-01-02 05:00:00,SSIU_Polymer,238,1,238.0,...,219,19,0,0,AF6H6C05RL,"{""AF6H6C05RL"": {""S"": 219, ""M"": 19, ""L"": 0, ""O""...",,,,2026-02-04 10:04:16
4,CAPIC100,aoi100,B156HTN06,CF,812,2026-01-02 10:00:00,BM_Cover,4,1,4.0,...,3,1,0,0,AF6H6C06K3,"{""AF6H6C06K3"": {""S"": 3, ""M"": 1, ""L"": 0, ""O"": 0...",,,,2026-02-04 10:04:16
